# 🐢 KNN шаг за шагом — медленная версия

Здесь **каждая строка кода объяснена**. Не спеши: прочитай комментарий (текст после `#`), потом запусти ячейку кнопкой ▶ или **Shift+Enter**.

Если что-то непонятно — подними руку, это нормально. Мы идём маленькими шагами.

---
**Что мы построим:** модель, которая по данным пассажира угадывает — выжил он на «Титанике» или нет.

## Шаг 1. Подключаем инструменты
Библиотеки — это готовые наборы команд. Подключаем те, что нужны.

In [ ]:
import pandas as pd   # pandas — работа с таблицами (зовём его "pd")

print("Готово! Pandas подключён.")

## Шаг 2. Загружаем таблицу с пассажирами
Данные лежат в интернете. Команда `read_csv` скачивает их в переменную `df` (это наша таблица).

In [ ]:
# адрес файла с данными
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

df = pd.read_csv(url)   # читаем таблицу в переменную df

df.head()   # показываем первые 5 строк, чтобы увидеть данные

## Шаг 3. Оставляем только нужные столбцы
В таблице много столбцов. Возьмём 4 признака + ответ. Лишнее уберём, чтобы не запутаться.

In [ ]:
# квадратные скобки [...] — список столбцов, которые оставляем
df = df[["Pclass", "Sex", "Age", "Fare", "Survived"]]

df.head()   # смотрим — теперь только 5 столбцов

## Шаг 4. Убираем пустые ячейки
У некоторых пассажиров не записан возраст. Пустые строки мешают модели — удалим их.

In [ ]:
df = df.dropna()   # dropna = "drop NA" = выбросить строки с пустотами

print("Осталось строк:", len(df))   # len() считает количество строк

## Шаг 5. Превращаем текст в числа
В столбце `Sex` написано "male"/"female". Модель понимает только числа.
Договоримся: мужчина = 1, женщина = 0.

In [ ]:
# (df["Sex"] == "male") даёт True/False, а .astype(int) превращает в 1/0
df["Sex"] = (df["Sex"] == "male").astype(int)

df.head()   # теперь в столбце Sex стоят 1 и 0

## Шаг 6. Делим таблицу на X и y
- **X** — признаки, ПО которым угадываем (4 столбца слева)
- **y** — ответ, который угадываем (столбец Survived)

In [ ]:
# X — это 4 столбца-признака
X = df[["Pclass", "Sex", "Age", "Fare"]]

# y — это столбец с ответом (выжил=1 / не выжил=0)
y = df["Survived"]

print("Признаки X — это столбцы:", list(X.columns))
print("Ответ y — это столбец: Survived")

## Шаг 7. Выравниваем числа (масштабирование)
Проблема: цена билета бывает 0–500, а возраст 0–80. Большие числа «перекричат» маленькие.
`StandardScaler` приводит все признаки к одному масштабу — честно для всех.

In [ ]:
from sklearn.preprocessing import StandardScaler   # инструмент выравнивания

scaler = StandardScaler()              # создаём "выравниватель"
X = scaler.fit_transform(X)            # применяем его к признакам

print("Готово! Теперь все признаки в одном масштабе.")

## Шаг 8. Создаём модель KNN
Говорим: «спрашивай 5 ближайших соседей». 5 — это наше число K (нечётное, чтобы не было ничьей).

In [ ]:
from sklearn.neighbors import KNeighborsClassifier   # сам алгоритм KNN

model = KNeighborsClassifier(n_neighbors=5)   # модель: K = 5 соседей

print("Модель создана. Пока она пустая — ещё ничему не научена.")

## Шаг 9. Обучаем модель 🎉
Команда `.fit()` показывает модели данные. KNN просто запоминает всех пассажиров.

In [ ]:
model.fit(X, y)   # fit = "обучить": модель запоминает X и ответы y

print("Модель обучена!")

## Шаг 10. Проверяем, как хорошо она угадывает
Просим модель предсказать ответы и сравниваем с правильными.

In [ ]:
from sklearn.metrics import accuracy_score   # инструмент подсчёта точности

predictions = model.predict(X)            # модель предсказывает ответы
accuracy = accuracy_score(y, predictions) # сравниваем с правдой

print("Точность:", round(accuracy, 2))    # round = округлить до 2 знаков

## Шаг 11. Предсказываем нового пассажира 🔮
Придумаем человека и спросим модель: выжил бы он?
Порядок чисел: [класс, пол, возраст, цена билета].

In [ ]:
# 1 класс, женщина (0), 25 лет, билет за 100
new_person = pd.DataFrame([[1, 0, 25, 100]],
                          columns=["Pclass", "Sex", "Age", "Fare"])

new_person = scaler.transform(new_person)   # выравниваем так же, как обучающие данные

answer = model.predict(new_person)          # модель решает

# если ответ 1 — выжил, если 0 — нет
print("Ответ модели:", "ВЫЖИЛ ✅" if answer[0] == 1 else "НЕ выжил ❌")

## ✍️ Твоё задание (по силам каждому)

**Уровень 1 (для всех):**
Поменяй числа в Шаге 11 — придумай своего пассажира и узнай его судьбу.
Например: 3 класс, мужчина (1), 40 лет, дешёвый билет (10).

**Уровень 2:**
В Шаге 8 поменяй `n_neighbors=5` на другое нечётное число (3, 7, 9).
Перезапусти шаги 8–10. Изменилась точность?

**Уровень 3 (для быстрых):**
Сделай 3 разных пассажиров и сравни их судьбы. Кто выжил, кто нет? Почему, как думаешь?

---
**Что мы сделали:** загрузили данные → почистили → разделили на X и y →
выровняли → обучили KNN → проверили → предсказали нового человека.

Это **полный путь любой ML-модели**. Дальше алгоритмы будут другие, но шаги — те же.